In [1]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator

print("GPU available:", tf.config.list_physical_devices('GPU'))

2026-05-26 06:56:42.126758: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779778602.350734      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779778602.418396      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779778602.960705      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779778602.960742      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1779778602.960745      57 computation_placer.cc:177] computation placer alr

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [2]:
import os

base_dir = '/kaggle/input/datasets/xhlulu/140k-real-and-fake-faces/real_vs_fake/real-vs-fake'
for split in ['train', 'valid','test']:
    for label in ['real', 'fake']:
        path = os.path.join(base_dir, split, label)
        count = len(os.listdir(path))
        print(f"{split}/{label} : {count} anhh")

train/real : 50000 anhh
train/fake : 50000 anhh
valid/real : 10000 anhh
valid/fake : 10000 anhh
test/real : 10000 anhh
test/fake : 10000 anhh


In [3]:
IMG_SIZE = 224
BATCH_SIZE = 32

# Augmentation cho train 
train_datagen = ImageDataGenerator(
    rescale=1./255, # normalize pixel ve [0,1]
    rotation_range = 20, #xoay toi da 30 do
    width_shift_range=0.1,#dich ngang toi da 20
    height_shift_range=0.1,# dich doc toi da 20
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip = True,
    brightness_range=[0.8,1.2],
    fill_mode='nearest' #fill pixel thieu bang nearest neighbor
)

#valid/test chi normalize, khong augment
val_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    os.path.join(base_dir, 'train'),
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'binary' #real 1, fake =0
)

val_gen = val_datagen.flow_from_directory(
    os.path.join(base_dir, 'valid'),
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'binary'
)

test_gen = val_datagen.flow_from_directory(
    os.path.join(base_dir, 'test'),
    target_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE,
    class_mode = 'binary',
    shuffle=False
)

print("Train:", train_gen.samples,"anh")
print("Valid:", val_gen.samples,"anh")
print("Class indices:", train_gen.class_indices)

Found 100000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Found 20000 images belonging to 2 classes.
Train: 100000 anh
Valid: 20000 anh
Class indices: {'fake': 0, 'real': 1}


In [5]:
def build_cnn_block(x, filters, dropout_rate=0.25):
    """
    1 conv block = Conv2D → BatchNorm → ReLU → MaxPool → Dropout
    filters: số filter tăng dần 32→64→128→256
    """
    x = layers.Conv2D(
        filters,
        kernel_size=(3, 3),
        padding='same',
        kernel_initializer='he_normal',   # paper dùng He normal init
        kernel_regularizer=tf.keras.regularizers.l2(1e-4)  # L2 reg
    )(x)
    x = layers.BatchNormalization()(x)    # ổn định training
    x = layers.Activation('relu')(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)  # giảm spatial dim
    x = layers.Dropout(dropout_rate)(x)
    return x

# Build full CNN backbone
inputs = layers.Input(shape=(224, 224, 3))

x = build_cnn_block(inputs, filters=32)   # 224→112
x = build_cnn_block(x, filters=64)        # 112→56
x = build_cnn_block(x, filters=128)       # 56→28
x = build_cnn_block(x, filters=256)       # 28→14

print("Shape sau 4 conv blocks:", x.shape)
# Kết quả mong đợi: (None, 14, 14, 256)

Shape sau 4 conv blocks: (None, 14, 14, 256)


In [6]:
def build_mhsa_block(x):
    """
    Sau conv blocks: Reshape → Dropout → MHSA → Residual → LayerNorm
    Đây là phần paper gọi là 'sequential integration'
    """
    # Bước 1: Reshape từ (14, 14, 256) → (196, 256)
    # MHSA cần input dạng sequence, không phải spatial grid
    seq_len = x.shape[1] * x.shape[2]   # 14×14 = 196
    x_reshaped = layers.Reshape((seq_len, 256))(x)  # (None, 196, 256)

    # Bước 2: Dropout trước attention (paper dùng 0.3)
    x_drop = layers.Dropout(0.3)(x_reshaped)

    # Bước 3: Multi-Head Self-Attention
    # num_heads=4, key_dim=64 — đúng theo paper Table 2
    attn_output = layers.MultiHeadAttention(
        num_heads=4,
        key_dim=64,
        dropout=0.2          # dropout bên trong attention (paper: 0.2)
    )(x_drop, x_drop)        # self-attention: query=key=value=x_drop

    # Bước 4: Residual connection — cộng input gốc vào output attention
    # Giúp giữ lại thông tin low-level từ CNN, tránh mất mát
    x_residual = layers.Add()([x_reshaped, attn_output])

    # Bước 5: Layer Normalization
    x_norm = layers.LayerNormalization()(x_residual)

    return x_norm

# Gắn MHSA vào sau CNN backbone
x = build_mhsa_block(x)

# Global Average Pooling: (196, 256) → (256,)
# Gộp toàn bộ sequence thành 1 vector đặc trưng
x = layers.GlobalAveragePooling1D()(x)

# Dense layers — phân loại cuối
x = layers.Dense(128)(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

x = layers.Dense(64)(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

# Output layer: sigmoid vì binary classification
outputs = layers.Dense(1, activation='sigmoid')(x)

# Tạo model
model = Model(inputs=inputs, outputs=outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 224, 224,  │        896 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 224, 224,  │        128 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_4        │ (None, 224, 224,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 112, 112,  │          0 │ activation_4[0][… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 112, 112,  │          0 │ max_pooling2d_4[… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 112, 112,  │     18,496 │ dropout_4[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 112, 112,  │        256 │ conv2d_5[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_5        │ (None, 112, 112,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 56, 56,    │          0 │ activation_5[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 56, 56,    │          0 │ max_pooling2d_5[… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 56, 56,    │     73,856 │ dropout_5[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 56, 56,    │        512 │ conv2d_6[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_6        │ (None, 56, 56,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_6     │ (None, 28, 28,    │          0 │ activation_6[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 28, 28,    │          0 │ max_pooling2d_6[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 28, 28,    │    295,168 │ dropout_6[0][0] 

 Total params: 696,001 (2.66 MB)

 Trainable params: 694,657 (2.65 MB)

 Non-trainable params: 1,344 (5.25 KB)

In [7]:
import tensorflow.keras.backend as K

def focal_loss(alpha=0.25, gamma=2.0):
    """
    Focal Loss — paper dùng alpha=0.25, gamma=2.0
    
    Tại sao không dùng Binary Cross-Entropy thông thường?
    BCE phạt mọi sample như nhau. Focal Loss phạt nặng hơn
    những sample model đang predict sai với confidence cao.
    gamma=2 nghĩa là sample dễ (confidence cao, đúng) 
    sẽ bị giảm loss đi rất nhiều → model tập trung vào hard samples.
    """
    def loss_fn(y_true, y_pred):
        y_pred = K.clip(y_pred, 1e-7, 1 - 1e-7)  # tránh log(0)
        
        # Binary cross entropy cơ bản
        bce = -y_true * K.log(y_pred) - (1 - y_true) * K.log(1 - y_pred)
        
        # Tính p_t: xác suất predict đúng class
        p_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        
        # Focal weight: (1 - p_t)^gamma
        # p_t cao (dễ) → weight gần 0 → loss nhỏ
        # p_t thấp (khó) → weight gần 1 → loss giữ nguyên
        focal_weight = K.pow(1 - p_t, gamma)
        
        # Alpha cân bằng 2 class
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        
        return K.mean(alpha_t * focal_weight * bce)
    
    return loss_fn

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss=focal_loss(alpha=0.25, gamma=2.0),
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

print("Model compiled OK")

Model compiled OK


In [ ]:
CHECKPOINT_PATH = '/kaggle/working/best_model.keras'
BACKUP_PATH = '/kaggle/working/backup_epoch.keras'

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=7,
        restore_best_weights=True,
        mode='max', verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', factor=0.5,
        patience=3, min_lr=1e-6,
        mode='max', verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        CHECKPOINT_PATH, monitor='val_auc',
        save_best_only=True, mode='max', verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        BACKUP_PATH, save_best_only=False, verbose=0
    )
]

history = model.fit(
    train_gen,
    epochs=10,          # chỉ train đến epoch 10
    initial_epoch=0,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print(f"\nBest val_auc: {max(history.history['val_auc']):.4f}")
print(f"Best val_accuracy: {max(history.history['val_accuracy']):.4f}")

In [ ]:
# ============================================
START_EPOCH = 10   # thay số này mỗi lần
END_EPOCH = 20     # thay số này mỗi lần
# ============================================

CHECKPOINT_PATH = '/kaggle/working/best_model.keras'
BACKUP_PATH = '/kaggle/working/backup_epoch.keras'

# Load best model
model = tf.keras.models.load_model(
    '/kaggle/input/models/manht05/last/keras/default/1/best_model.keras',
    custom_objects={'loss_fn': focal_loss()}
)

# Tính lr giảm dần theo số lần chạy
lr = 1e-4 * (0.8 ** (START_EPOCH // 10))
print(f"Resume epoch {START_EPOCH}→{END_EPOCH}, lr={lr:.6f}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
    loss=focal_loss(alpha=0.25, gamma=2.0),
    metrics=['accuracy',
             tf.keras.metrics.AUC(name='auc'),
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall')]
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_auc', patience=7,
        restore_best_weights=True,
        mode='max', verbose=1
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_auc', factor=0.5,
        patience=3, min_lr=1e-6,
        mode='max', verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        CHECKPOINT_PATH, monitor='val_auc',
        save_best_only=True, mode='max', verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        BACKUP_PATH, save_best_only=False, verbose=0
    )
]

history = model.fit(
    train_gen,
    epochs=END_EPOCH,
    initial_epoch=START_EPOCH,
    validation_data=val_gen,
    callbacks=callbacks,
    verbose=1
)

print(f"\n--- Epoch {START_EPOCH}→{END_EPOCH} xong ---")
print(f"Best val_auc:      {max(history.history['val_auc']):.4f}")
print(f"Best val_accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Nhớ Save Version trước khi tắt!")

Resume epoch 10→20, lr=0.000080


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 11/20


I0000 00:00:1779778789.822746     154 service.cc:152] XLA service 0x7ce32c10c0a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1779778789.822796     154 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1779778789.822802     154 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1779778790.999150     154 cuda_dnn.cc:529] Loaded cuDNN version 91002


   1/3125 ━━━━━━━━━━━━━━━━━━━━ 18:53:28 22s/step - accuracy: 0.6250 - auc: 0.8393 - loss: 0.0812 - precision: 0.8000 - recall: 0.4444

I0000 00:00:1779778804.689094     154 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 602ms/step - accuracy: 0.7966 - auc: 0.9122 - loss: 0.0516 - precision: 0.9082 - recall: 0.6589
Epoch 11: val_auc improved from -inf to 0.90919, saving model to /kaggle/working/best_model.keras
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 2052s 650ms/step - accuracy: 0.7966 - auc: 0.9122 - loss: 0.0516 - precision: 0.9082 - recall: 0.6589 - val_accuracy: 0.8267 - val_auc: 0.9092 - val_loss: 0.0573 - val_precision: 0.8465 - val_recall: 0.7981 - learning_rate: 8.0000e-05
Epoch 12/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 443ms/step - accuracy: 0.8065 - auc: 0.9211 - loss: 0.0496 - precision: 0.9136 - recall: 0.6762
Epoch 12: val_auc did not improve from 0.90919
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 1439s 460ms/step - accuracy: 0.8065 - auc: 0.9211 - loss: 0.0496 - precision: 0.9136 - recall: 0.6762 - val_accuracy: 0.8162 - val_auc: 0.8980 - val_loss: 0.0630 - val_precision: 0.8219 - val_recall: 0.8073 - learning_rate: 8.0000e-05
Epoch 13/20
3125/3125 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms